# ShootPX — On-Model Shots: fal.ai Provider Test Notebook

**Goal of this notebook:** simulate the exact flow your production UI will offer, end-to-end,
against real fal.ai models — so you pick a winning provider *before* any of this becomes
`app/core/fal_provider.py` / `on_model_shots.py` code.

## The user-facing flow this notebook mirrors

1. **Model input** — user picks one of:
   - `generate` — describe a model in text, we text-to-image it first
   - `upload` — user provides their own model photo
   - `default` — pick from a preset model library (simulated here with local files/URLs)
2. **Garment input** — one or more product photos (front/back/detail angles all allowed)
3. **Reference images** *(optional)* — extra images purely for the model to understand
   style/pose/mood, not garment identity
4. **Output settings** — number of images, resolution, aspect ratio
5. **Prompt mode**:
   - **User-supplied prompt** → skip reasoning, generate directly (1 call per requested image)
   - **No prompt supplied** → a VLM ("prompt writer") looks at the model + garment + refs and
     writes **N distinct pose prompts** (front / back-¾ / side / detail close-up), then we run
     **one generation call per pose** — this is required because a single prompt run N times
     gives sampling variation, not controlled pose diversity
6. **Generation** — swappable provider function
7. **Safety** — `enable_safety_checker` stays `True` throughout; a light local pre-check runs
   before any request goes out

## Model candidates being evaluated

All five below were checked live against fal.ai's own docs/catalog on **2026-08-29** — slugs
and schemas drift, re-verify if you run this much later.

| Family | Model | Notes |
|---|---|---|
| Prompt-driven edit | `fal-ai/bytedance/seedream/v4.5/edit` | Fully wired. Aspect ratio via named **preset** (`image_size`), not a raw ratio — the odd one out. |
| Prompt-driven edit | `fal-ai/flux-pro/kontext/max/multi` | Fully wired. **Not** `fal-ai/flux-pro/kontext` (plain) — that variant only takes one `image_url` and can't hold model+garment together. |
| Prompt-driven edit | `fal-ai/gemini-25-flash-image/edit` (aka `fal-ai/nano-banana/edit`) | Fully wired. Identical schema under either slug. |
| Purpose-built try-on | `fal-ai/idm-vton` | Fully wired. Still live but no longer listed on fal's own `/explore/virtual-try-on-apis` — may be legacy. Output is a **single `image` object**, not an `images` list (the obvious guess is wrong here). |
| Purpose-built try-on | `fal-ai/fashn/tryon/v1.6` | Added — the try-on model fal's current catalog actually surfaces. Different field names again (`model_image`/`garment_image`, singular). |

Family A (prompt-driven) shares one `(prompt, image_urls) -> [url]` shape and runs through
`run_full_flow()`. Family B (purpose-built try-on) has no prompt, no pose control, and no
aspect-ratio input at all — it gets its own mini-harness, `run_purpose_built_tryon()`, further
down. Run the same model + garment + settings through each candidate, score them with the
rubric at the bottom, then hand the winning slug + payload back for the real integration.


## 0. Setup

In [ ]:
# pip install fal-client anthropic pillow requests python-dotenv

import os
import io
import json
import time
import requests
from pathlib import Path
from PIL import Image
from IPython.display import display

import fal_client

# Set these in your environment, or uncomment and paste for local testing only
# (never commit real keys)
# os.environ["FAL_KEY"] = "your-fal-key"
# os.environ["ANTHROPIC_API_KEY"] = "your-anthropic-key"

assert os.environ.get("FAL_KEY"), "Set FAL_KEY before running."


## 1. Upload helper

Every fal model expects **URLs**, not raw bytes. Local files go through `fal_client.upload_file`
first. If you already have a URL (e.g. a product photo already on your CDN), it passes through
unchanged.


In [ ]:
def to_hosted_url(path_or_url: str) -> str:
    """Return a fal-hosted URL for a local file, or pass a URL through unchanged."""
    if path_or_url.startswith("http://") or path_or_url.startswith("https://"):
        return path_or_url
    return fal_client.upload_file(path_or_url)


def show(url_or_path: str, caption: str = ""):
    if url_or_path.startswith("http"):
        img = Image.open(io.BytesIO(requests.get(url_or_path, timeout=30).content))
    else:
        img = Image.open(url_or_path)
    print(caption)
    display(img)


## 2. Step 1 — Model input

Simulates the three UI options. `default` pulls from a small local preset list you populate —
this is your future "Model Library" tab.


In [ ]:
MODEL_LIBRARY = {
    # "preset_id": "path/or/url/to/model.jpg"
    "preset_1": "assets/models/preset_1.jpg",
    "preset_2": "assets/models/preset_2.jpg",
}

# Text-to-image slug for the "generate a model" path.
# NOTE: verify the current live slug on fal.ai before running — this drifts.
TEXT_TO_IMAGE_MODEL = "fal-ai/bytedance/seedream/v4.5/text-to-image"  # <-- VERIFY ON FAL.AI


def generate_model_via_text2image(description: str, image_size="portrait_4_3") -> str:
    """Text-to-image a base model photo to use as the 'upload' input downstream."""
    prompt = (
        f"Photorealistic studio portrait of {description}. Neutral relaxed standing pose, "
        f"plain light-gray studio background, soft even lighting, natural skin texture, "
        f"looking at camera, commercial fashion-catalog quality, high detail."
    )
    result = fal_client.subscribe(
        TEXT_TO_IMAGE_MODEL,
        arguments={"prompt": prompt, "image_size": image_size, "num_images": 1},
        with_logs=True,
    )
    return result["images"][0]["url"]


def resolve_model_image(mode: str, value: str) -> str:
    """
    mode: "generate" | "upload" | "default"
    value: text description (generate) | local path or URL (upload) | preset_id (default)
    """
    if mode == "generate":
        return generate_model_via_text2image(value)
    elif mode == "upload":
        return to_hosted_url(value)
    elif mode == "default":
        return to_hosted_url(MODEL_LIBRARY[value])
    raise ValueError(f"Unknown model mode: {mode}")


## 3. Step 2 — Garment + reference images

Multiple garment angles are supported (Seedream accepts up to **10** input images per call,
and **total input + output images per call must stay ≤ 15**). We keep an explicit ordered
label map so every prompt we write can say exactly "Image 2 is the garment front" etc.,
instead of relying on the model to guess.


In [ ]:
def assemble_inputs(model_image, garment_images, reference_images=None):
    """
    Returns (image_urls, labels) where labels[i] describes image_urls[i] in order.
    Enforces fal's input-count ceiling.
    """
    reference_images = reference_images or []
    ordered = [("model", model_image)]
    ordered += [("garment", g) for g in garment_images]
    ordered += [("reference", r) for r in reference_images]

    if len(ordered) > 10:
        raise ValueError("Max 10 input images per Seedream call — trim garment/reference count.")

    image_urls = [to_hosted_url(url) for _, url in ordered]
    labels = [
        f"Image {i+1} = {kind} ({'model reference' if kind=='model' else 'exact product, preserve fidelity' if kind=='garment' else 'style/pose reference only'})"
        for i, (kind, _) in enumerate(ordered)
    ]
    return image_urls, labels


## 4. Step 3 — Output settings (resolution + aspect ratio)

Confirmed from the Seedream 4.5 edit schema: `image_size` accepts either a **named preset**
(covers both aspect ratio *and* a resolution tier) or a **custom `{width, height}`** object.
No separate "aspect ratio" field exists — the preset name *is* the aspect ratio.

| Preset | Aspect ratio | Use case |
|---|---|---|
| `square_hd` / `square` | 1:1 | Marketplace thumbnail |
| `portrait_4_3` | 3:4 | Standard product/model shot |
| `portrait_16_9` | 9:16 | Reels / vertical social |
| `landscape_4_3` | 4:3 | Banner |
| `landscape_16_9` | 16:9 | Wide banner |
| `auto_2K` / `auto_4K` | model decides | Resolution-priority, not ratio-priority |

Map your UI's aspect-ratio pills directly to this table. If a user wants a specific pixel size,
pass `{"width": W, "height": H}` instead (must be 1920–4096 per axis, or 2.56M–16.8M total px).


In [ ]:
ASPECT_RATIO_MAP = {
    "1:1": "square_hd",
    "3:4": "portrait_4_3",
    "9:16": "portrait_16_9",
    "4:3": "landscape_4_3",
    "16:9": "landscape_16_9",
}

def build_image_size(mode: str, value):
    """
    mode: "aspect_ratio" | "resolution_auto" | "custom"
    value: "3:4" | "4K"/"2K" | {"width":.., "height":..}
    """
    if mode == "aspect_ratio":
        return ASPECT_RATIO_MAP[value]
    if mode == "resolution_auto":
        return "auto_4K" if value.upper() == "4K" else "auto_2K"
    if mode == "custom":
        w, h = value["width"], value["height"]
        assert 1920 <= w <= 4096 and 1920 <= h <= 4096, "Width/height must be 1920-4096"
        return {"width": w, "height": h}
    raise ValueError(f"Unknown image_size mode: {mode}")


## 5. Step 4 — Prompt logic

Two paths:

- **User supplies a prompt** → used as-is, one generation call (looped `num_images` times if
  you want literal repeats — but note this gives *sampling* variation, not different poses).
- **No prompt supplied** → a VLM writes N distinct, Seedream-native prompts (short, directive,
  one per pose), each run as its own generation call. This is the "for the model + garment,
  give me 4 different poses" behavior you described.

The system instruction below encodes the concise, directive prompting style confirmed against
fal's own documented examples (short imperative sentences, explicit image referencing, garment
fidelity stated once — not the long GPT-style megaprompt pattern).


In [ ]:
import anthropic

claude = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from env

PROMPT_WRITER_SYSTEM = """You write short, directive image-edit prompts for Seedream 4.5, an \
instruction-following image editing model — not a scene-description model. Follow its native \
style: concise imperative sentences, explicit numbered image references (Image 1, Image 2, ...), \
state garment-preservation requirements once, do not pad with repeated adjectives or long \
negative-instruction lists. Each prompt must specify a distinct, named pose so a batch of \
prompts produces genuinely different shots, not just resampled variations of the same pose."""

def generate_pose_prompts_via_vlm(image_urls, labels, num_poses=4, garment_type="bra"):
    poses = ["front-facing, shoulders squared to camera",
             "back three-quarter turn, showing strap and band construction",
             "side profile, three-quarter body turn",
             "close-up detail shot of the garment on the model, waist-up"][:num_poses]

    content = [{"type": "text", "text":
        f"Images in order:\n" + "\n".join(labels) +
        f"\n\nWrite {num_poses} separate Seedream-native prompts for a premium ecommerce "
        f"fashion catalog shoot of this {garment_type} on this model. One prompt per pose, "
        f"using exactly this pose list in order: {poses}. "
        f"Return ONLY a JSON array of {num_poses} strings, nothing else."}]
    for url in image_urls:
        content.append({"type": "image", "source": {"type": "url", "url": url}})

    resp = claude.messages.create(
        model="claude-opus-5",
        max_tokens=1500,
        system=PROMPT_WRITER_SYSTEM,
        messages=[{"role": "user", "content": content}],
    )
    text = resp.content[0].text
    return json.loads(text)


## 6. Step 5 — Generation: Seedream 4.5 edit (fully wired, real schema)


In [ ]:
SEEDREAM_EDIT_MODEL = "fal-ai/bytedance/seedream/v4.5/edit"

def run_seedream_edit(prompt, image_urls, aspect_ratio="3:4", num_images=1, seed=None):
    # Seedream wants a named PRESET, not a raw "W:H" ratio — translate through
    # ASPECT_RATIO_MAP. Anything not in the map (e.g. "auto_4K"/"auto_2K", or a
    # {"width","height"} dict) passes through unchanged, so those still work directly.
    image_size = ASPECT_RATIO_MAP.get(aspect_ratio, aspect_ratio)
    args = {
        "prompt": prompt,
        "image_urls": image_urls,
        "image_size": image_size,
        "num_images": num_images,
        "max_images": 1,
        "enable_safety_checker": True,  # do not disable
    }
    if seed is not None:
        args["seed"] = seed

    result = fal_client.subscribe(
        SEEDREAM_EDIT_MODEL,
        arguments=args,
        with_logs=True,
    )
    return [img["url"] for img in result["images"]]


## 7. Step 5b — the remaining candidates (verified schemas, 2026-08-29)

**Aspect ratio note:** Flux Kontext Multi and Gemini 2.5 Flash Image both take a plain
`aspect_ratio` string ("3:4", "9:16", "1:1", ...) that matches `ASPECT_RATIO_MAP`'s keys
directly — no translation needed. Seedream is the odd one out (named preset via `image_size`,
handled inside `run_seedream_edit`). idm-vton and FASHN have **no** size/ratio control at all —
output dimensions are whatever the model decides from the inputs.


In [ ]:
# --- Family A: prompt-driven edit models — same (prompt, image_urls) -> [url] shape,
# so these drop straight into run_full_flow() via the generator_fn= swap. ---

def run_flux_kontext(prompt, image_urls, aspect_ratio="3:4", num_images=1, seed=None, **kwargs):
    # "max/multi" variant specifically — plain fal-ai/flux-pro/kontext takes only ONE
    # image_url, which can't hold model+garment together.
    model_id = "fal-ai/flux-pro/kontext/max/multi"
    args = {
        "prompt": prompt,
        "image_urls": image_urls,
        "aspect_ratio": aspect_ratio,
        "num_images": num_images,
    }
    if seed is not None:
        args["seed"] = seed
    result = fal_client.subscribe(model_id, arguments=args, with_logs=True)
    return [img["url"] for img in result["images"]]


def run_nano_banana(prompt, image_urls, aspect_ratio="3:4", num_images=1, seed=None, **kwargs):
    # Same endpoint under two aliases (fal-ai/gemini-25-flash-image/edit,
    # fal-ai/nano-banana/edit) — identical schema either way.
    model_id = "fal-ai/gemini-25-flash-image/edit"
    args = {
        "prompt": prompt,
        "image_urls": image_urls,
        "aspect_ratio": aspect_ratio,
        "num_images": num_images,
    }
    if seed is not None:
        args["seed"] = seed
    result = fal_client.subscribe(model_id, arguments=args, with_logs=True)
    return [img["url"] for img in result["images"]]


# --- Family B: purpose-built try-on models — no prompt, no pose control (pose comes
# from the model photo itself), no aspect-ratio input. Different shape entirely, so
# these do NOT plug into run_full_flow()'s generator_fn= slot — use
# run_purpose_built_tryon() below instead. ---

def run_idm_vton(human_image_url, garment_image_url, description, seed=None, **kwargs):
    model_id = "fal-ai/idm-vton"
    args = {
        "human_image_url": human_image_url,
        "garment_image_url": garment_image_url,
        "description": description,
    }
    if seed is not None:
        args["seed"] = seed
    result = fal_client.subscribe(model_id, arguments=args, with_logs=True)
    # Real output is a single `image` object, not an `images` list.
    return [result["image"]["url"]]


def run_fashn_tryon(human_image_url, garment_image_url, category="auto", num_samples=1, **kwargs):
    # The try-on model fal's own /explore/virtual-try-on-apis catalog currently surfaces.
    # Field names differ again: model_image / garment_image, both singular.
    model_id = "fal-ai/fashn/tryon/v1.6"
    args = {
        "model_image": human_image_url,
        "garment_image": garment_image_url,
        "category": category,
        "num_samples": num_samples,
    }
    result = fal_client.subscribe(model_id, arguments=args, with_logs=True)
    return [img["url"] for img in result["images"]]


def run_purpose_built_tryon(model_image, garment_image, description="", generator_fn=run_idm_vton, **kwargs):
    """Separate mini-harness for Family B: one call in, one image out, no VLM step,
    no pose looping. Swap generator_fn=run_fashn_tryon to compare against idm-vton."""
    model_url = to_hosted_url(model_image)
    garment_url = to_hosted_url(garment_image)
    if generator_fn is run_fashn_tryon:
        urls = generator_fn(model_url, garment_url, **kwargs)
    else:
        urls = generator_fn(model_url, garment_url, description, **kwargs)
    for u in urls:
        show(u, caption=generator_fn.__name__)
    return urls


## 8. Step 6 — Light local safety pre-check

This is **not** a replacement for `enable_safety_checker=True` (which stays on for every real
call above) — it's a cheap local gate before you even spend a request, e.g. catching obviously
wrong inputs before they hit the API.


In [ ]:
BLOCKED_TERMS = ["child", "minor", "teen", "kid", "underage"]  # extend as needed

def local_safety_precheck(model_description: str, garment_type: str):
    text = f"{model_description} {garment_type}".lower()
    for term in BLOCKED_TERMS:
        if term in text:
            raise ValueError(f"Blocked term detected in description: '{term}'. Request not sent.")
    return True


## 9. End-to-end test harness

Wire steps 1–6 together exactly like the production flow will.


In [ ]:
def run_full_flow(
    model_mode, model_value,             # e.g. "upload", "path/to/model.jpg"
    garment_paths,                       # list of local paths or URLs
    reference_paths=None,                # optional
    aspect_ratio="3:4",
    num_poses=4,
    user_prompt=None,                    # if provided, skips VLM step
    garment_type="bra",
    generator_fn=run_seedream_edit,      # swap to run_flux_kontext / run_nano_banana to A/B test
                                          # (Family B — run_idm_vton / run_fashn_tryon — has a
                                          # different shape; use run_purpose_built_tryon() instead)
):
    local_safety_precheck(model_value if model_mode == "generate" else "uploaded model", garment_type)

    model_image = resolve_model_image(model_mode, model_value)
    image_urls, labels = assemble_inputs(model_image, garment_paths, reference_paths)

    if user_prompt:
        prompts = [user_prompt] * num_poses
    else:
        prompts = generate_pose_prompts_via_vlm(image_urls, labels, num_poses=num_poses, garment_type=garment_type)

    outputs = []
    for i, p in enumerate(prompts):
        print(f"--- Pose {i+1}/{len(prompts)} ---\nPrompt: {p}\n")
        urls = generator_fn(p, image_urls, aspect_ratio=aspect_ratio, num_images=1)
        outputs.extend(urls)
        for u in urls:
            show(u, caption=f"Pose {i+1}")

    return {"prompts": prompts, "image_urls": image_urls, "labels": labels, "outputs": outputs}


In [ ]:
# Example run — fill in real paths/URLs before executing
# result = run_full_flow(
#     model_mode="upload",
#     model_value="assets/model_ref.jpg",
#     garment_paths=["assets/bra_front.jpg", "assets/bra_back.jpg"],
#     reference_paths=None,
#     aspect_ratio="3:4",
#     num_poses=4,
#     user_prompt=None,   # None => VLM writes 4 distinct pose prompts
#     generator_fn=run_seedream_edit,   # or run_flux_kontext / run_nano_banana
# )

# Family B (purpose-built try-on) — separate harness, no prompt/pose args:
# tryon_urls = run_purpose_built_tryon(
#     model_image="assets/model_ref.jpg",
#     garment_image="assets/bra_front.jpg",
#     description="a red cotton bra",   # only used by idm-vton; ignored by FASHN
#     generator_fn=run_idm_vton,        # or run_fashn_tryon
# )


## 10. Scoring rubric — fill this in per model tested

Run the same inputs through `run_seedream_edit`, `run_flux_kontext`, `run_nano_banana` (via
`run_full_flow`), and `run_idm_vton`, `run_fashn_tryon` (via `run_purpose_built_tryon`), then
score each 1–5.


In [ ]:
import pandas as pd

scorecard = pd.DataFrame(columns=[
    "model_slug", "garment_fidelity", "model_realism", "pose_control",
    "latency_sec", "cost_per_image_usd", "notes"
])

# scorecard.loc[len(scorecard)] = ["fal-ai/bytedance/seedream/v4.5/edit", 5, 4, 5, 39.0, 0.04, "baseline, validated"]

scorecard


## Next steps

Once a winner is picked here, bring back to the app-integration session:

1. The exact `model_id` slug that won
2. A working request payload + the response JSON shape (where the output image URL actually
   lives — confirmed above: `images[].url` for Seedream/Flux Kontext/nano-banana/FASHN, but
   `image.url` — singular, no `s` — for idm-vton)
3. Whether you want the real integration on blocking `subscribe()` or async `submit()`/`poll()`
   (the latter matches `GenerationHandle`/`poll_result()` in `ai_provider.py` already)
4. Final aspect-ratio handling you settled on: a shared `aspect_ratio` string works as-is for
   Flux Kontext Multi and nano-banana, needs the `ASPECT_RATIO_MAP` translation for Seedream,
   and has no effect on idm-vton/FASHN (no such input exists on those two)
